# ----------------------------------------------------------------------------------------
# EVALUATION FINAL - CARTOGRAPHIE ET SIG AVEC PYTHON - AGRIFORLAND
# -----------------------------------------------------------------------------------------

# 2.1 : Préparation de l’environnement
### a)  Installation des packages nécessaires

In [1]:
! pip install geopandas
! pip install geemap
! pip install earthengine-api
! pip install pandas
! pip install geedim rasterio matplotlib localtileserver ipyleaflet leafmap folium

In [1]:
pip install --upgrade ipywidgets ipykernel jupyterlab

  Using cached jupyterlab-4.5.7-py3-none-any.whl.metadata (16 kB)
  Using cached fqdn-1.5.1-py3-none-any.whl.metadata (1.4 kB)
  Using cached isoduration-20.11.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached jsonpointer-3.1.1-py3-none-any.whl.metadata (2.4 kB)
  Using cached rfc3987_syntax-1.1.0-py3-none-any.whl.metadata (7.7 kB)
  Using cached uri_template-1.3.0-py3-none-any.whl.metadata (8.8 kB)
  Using cached webcolors-25.10.0-py3-none-any.whl.metadata (2.2 kB)
  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached arrow-1.4.0-py3-none-any.whl.metadata (7.7 kB)
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/12.5 MB ? eta -:--:--
    --------------------------------------- 0.3/12.5 MB ? eta -:--:--
    --------------------------------------- 0.3/12.5 MB ? eta -:--:--
    --------------------------------------- 0.3/12.5 M

### b) Importation des bibliothèques

In [3]:
import os

from pathlib import Path
import json
import re
import warnings
import ee
import geemap
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
sns.set_theme(style="whitegrid")

# 2.2 : Chargement des données

### a) Définition des chemins de travail

In [4]:
# Dossier des données

data_folder = Path(r"C:/Users/HP/Desktop/evaluation_formation_python/Projet_evaluation_fin_formation/data")

# Fichiers

parcelles_file = data_folder / "parcelles_agricoles_cote_ivoire_400.geojson"
limite_file = data_folder / "Limite_de_la_Côte d_Ivoire.geojson"

# Dossier de sortie

output_folder = Path(r"C:/Users/HP/Desktop/evaluation_formation_python/Projet_evaluation_fin_formation/output")
output_folder.mkdir(parents=True, exist_ok=True)

html_file = output_folder / "carte_agricole_cote_ivoire.html"

print("Prêt.")

Prêt.


### b) Lecture de fichier GeoJSON

In [5]:
parcelles = gpd.read_file(parcelles_file)
limite = gpd.read_file(limite_file)

print("Parcelles :", len(parcelles))
print("Limite :", len(limite))

Parcelles : 386
Limite : 1


### c) Vérification des données

In [6]:
display(parcelles.head())
display(parcelles.columns)

,id_parcelle,nom_prenoms,date_naissance,superficie_ha,annee_creation_parcelle,speculation,production_tonne,zone_agricole,geometry
0,PARC_0001,Dago Kouadio,1969-04-28,4.68,1990,Cacao,4.59,Agboville,"POLYGON ((-4.33481 5.72113, -4.335 5.72041, -4..."
1,PARC_0002,N'Dri Akissi,1975-11-20,16.40,2007,Palmier à huile,92.01,Divo,"POLYGON ((-5.55491 5.77107, -5.55397 5.77152, ..."
2,PARC_0003,Koffi Issa,1990-12-28,12.92,1989,Hévéa,20.95,Man,"POLYGON ((-7.57443 7.35827, -7.5743 7.35724, -..."
3,PARC_0004,Koulibaly Eric,1960-09-12,13.89,2014,Palmier à huile,132.74,Gagnoa,"POLYGON ((-5.70049 6.00354, -5.70037 6.00247, ..."
4,PARC_0005,Traoré Souleymane,2000-07-18,4.96,1992,Cacao,5.90,San Pedro,"POLYGON ((-6.65807 4.79363, -6.65747 4.79335, ..."


Index(['id_parcelle', 'nom_prenoms', 'date_naissance', 'superficie_ha',
       'annee_creation_parcelle', 'speculation', 'production_tonne',
       'zone_agricole', 'geometry'],
      dtype='object')

# 2.3 : Création de carte interactive

### a) Initialiser Earth Engine

In [7]:
ee.Authenticate()
ee.Initialize()

### b) Création de la carte 

In [8]:
Map = geemap.Map(center=[7.6, -5.5], zoom=7)

Map.add_basemap("SATELLITE")
Map

Map(center=[7.6, -5.5], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright', t…

### c) Ajout limite Côte d’Ivoire

In [9]:
Map.add_gdf(
    limite,
    layer_name="Limite Côte d'Ivoire",
    style={
        "color": "black",
        "fillOpacity": 0
    }
)

# 2.4 : Ajout des parcelles agricoles

In [10]:
# Convertir toutes les colonnes dates en texte

for col in parcelles.columns:
    if str(parcelles[col].dtype).startswith("datetime"):
        parcelles[col] = parcelles[col].astype(str)

print("Dates converties avec succès.")

Dates converties avec succès.


In [11]:
Map.add_gdf(
    parcelles,
    layer_name="Parcelles agricoles",
    style={
        "color": "green",
        "fillColor": "green",
        "fillOpacity": 0.5
    },
    info_mode="on_click"
)

# 2.5 : Ajout d'une couche d’occupation du sol

In [12]:
worldcover = ee.Image("ESA/WorldCover/v100/2020")

vis = {
    "min": 10,
    "max": 100,
    "palette": [
        "006400",
        "ffbb22",
        "ffff4c",
        "f096ff",
        "fa0000",
        "b4b4b4",
        "f0f0f0"
    ]
}

Map.addLayer(worldcover, vis, "Occupation du sol")

# 2.6 : Ajout des interactions

### a) Ajout de Contrôle des couches

In [13]:
Map.addLayerControl()

### b) Ajout de légende

In [14]:
legend = {
    "Arbres": "006400",
    "Cultures": "ffbb22",
    "Prairie": "ffff4c",
    "Zones humides": "f096ff",
    "Bâtiments": "fa0000",
    "Roche": "b4b4b4"
}

Map.add_legend(title="Occupation du sol", legend_dict=legend)

# 2.7 : Exportation de la carte en HTML

In [14]:
Map.to_html(
    filename=str(html_file),
    title="Mini WebSIG Agricole Côte d'Ivoire",
    width="100%",
    height="900px"
)
print("Carte exportée avec succès :", html_file)

Carte exportée avec succès : C:\Users\HP\Desktop\evaluation_formation_python\Projet_evaluation_fin_formation\output\carte_agricole_cote_ivoire.html


#### Affichage de la carte finale

In [15]:
Map

Map(bottom=16030.0, center=[7.1663003819031825, -5.191040039062501], controls=(WidgetControl(options=['positio…

## 2.8 : Déployer
#### Publication sur GitHub Pages